# California Housing Prices

Se va a trabajar con el siguiente dataset https://www.kaggle.com/datasets/camnugent/california-housing-prices

| Variable | Significado |
|---|---|
| `longitude` | Coordenada geográfica oeste-este del barrio |
| `latitude` | Coordenada geográfica norte-sur del barrio |
| `housing_median_age` | Edad media de las casas en el barrio |
| `total_rooms` | Total de habitaciones en todo el barrio |
| `total_bedrooms` | Total de dormitorios en todo el barrio |
| `population` | Cantidad de personas que viven en el barrio |
| `households` | Número de hogares (familias) en el barrio |
| `median_income` | Ingreso medio de las familias del barrio |
| `median_house_value` | 🎯 TARGET — Precio medio de las casas en USD |
| `ocean_proximity` | Qué tan cerca está el barrio del océano |

Respondiendo las preguntas antes de realizar el modelo:

¿Cuál es el objetivo de negocio?
- El objetivo es construir un modelo para predecir el precio medio de viviendas en california con datos del censo.

¿Qué soluciones existen hoy?
- Supongamos que existen expertos inmobiliarios que utilizan excel.

¿Supervisado o no supervisado?
- Aprendizaje supervizado los datos ya estan etiquetados, es un problema de regresion ya que queremos predecir un valor númerico.

¿Cómo medimos el rendimiento?
- Para regresion tenemos el RMSE y el MAE.

## Instalación de paquetes necesarios

In [2]:
import os
import getpass
import json
import pandas as pd
import numpy as np

In [ ]:
print("Ingresa tus credenciales de Kaggle:")
username = getpass.getpass("👤 Username: ")
api_key  = getpass.getpass("🔑 API Key:  ")

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": username, "key": api_key}, f)

os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("✅ Credenciales configuradas")

In [ ]:
!pip install kaggle -q

In [ ]:
!kaggle datasets list -s "california housing"

ref                                                        title                                                 size  lastUpdated          downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  --------------------------------------------------  ------  -------------------  -------------  ---------  ---------------  
camnugent/california-housing-prices                        California Housing Prices                            400KB  2017-11-24 03:14:59         304306       1658  0.85294116       
harrywang/housing                                          California Housing Data (1990)                       400KB  2018-05-10 15:56:31          21477        189  1.0              
dhirajnirne/california-housing-data                        California Housing Data                              400KB  2021-05-15 21:21:05           6577         30  1.0              
shraddha4ever20/california-housing-dataset                 California Housing da

In [ ]:
!kaggle datasets download -d camnugent/california-housing-prices --unzip -p data/

Dataset URL: https://www.kaggle.com/datasets/camnugent/california-housing-prices
License(s): CC0-1.0
  0% 0.00/400k [00:00<?, ?B/s]
100% 400k/400k [00:00<00:00, 97.6MB/s]


In [ ]:
df = pd.read_csv("data/housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


Separamos los feature y target
- X = Features (lo que usamos para predecir)
- y = Target (lo que queremos predecir)

In [ ]:
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

X shape: (20640, 9)
y shape: (20640,)


In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   ocean_proximity     20640 non-null  object 
dtypes: float64(8), object(1)
memory usage: 1.4+ MB


## RMSE — Root Mean Squared Error

Mide el error promedio del modelo, castigando más los errores grandes. Es muy subsetible a los outlier por que pueden inflar el RMSE y hacerte creer que tu modelo es malo.

$$RMSE(X, y, h) = \sqrt{\frac{1}{m} \sum_{i=1}^{m}(h(x^{(i)}) - y^{(i)})^2}$$

Donde:
- $m$ es el total de datos de las predicciones a comparar.
- $h$ es la función del modelo
- $h(x^{(i)})$ es el resultado de la predicción
- $y^{(i)}$ es el valor real.



```
RMSE(X, y, h)
      │  │  │
      │  │  └── h = hipótesis = tu modelo = la función que predice
      │  └───── y = valores reales (target)
      └──────── X = features (datos de entrada)


h(x⁽ⁱ⁾) - y⁽ⁱ⁾
│             │
│             └── valor REAL de la fila i
└──────────────── lo que PREDIJO el modelo para la fila i
```



In [ ]:
def rmse(X, y, h):
  m = len(y)
  error_sum = 0

  for i in range(m):                        # recorre cada fila
        y_pred = h(X[i])                      # modelo predice para la fila i
        error_sum += (y_pred - y[i]) ** 2     # suma (predicción - real)²
  return np.sqrt(error_sum / m)

## MAE — Mean Absolute Error


Mide el error promedio del modelo de forma honesta, tratando todos los errores igual.
Es más robusto ante outliers porque no los castiga extra al no elevar al cuadrado.

$$MAE(X, y, h) = \frac{1}{m} \sum_{i=1}^{m}|h(x^{(i)}) - y^{(i)}|$$

Donde:
- $m$ es el total de datos de las predicciones a comparar.
- $h$ es la función del modelo
- $h(x^{(i)})$ es el resultado de la predicción
- $y^{(i)}$ es el valor real.
- $|\cdot|$ es el valor absoluto — convierte negativos en positivos

In [1]:
def mae(X, y, h):
    m = len(y)        # total de datos
    error_sum = 0     # acumulador

    for i in range(m):
        y_pred = h(X[i])                    # predicción del modelo
        error_sum += abs(y_pred - y[i])     # valor absoluto en vez de cuadrado

    return error_sum / m                    # promedio sin raíz cuadrada